# ポートリャーギンの最小原理(PMP)の導出

変分法との対比


|項目|変分法|PMP|
|:---|:---|:---|
|評価関数 |$J[y]$ | $J[x,u]$|
|最適化する対象|関数 $y(x)$ | 状態軌道 $x(t)$と入力軌道 $u(t)$ |
|制約条件|なし(基本形)|状態方程式 $\dot{x}=f(x,u)$|
|必要条件|オイラー・ラグランジュ方程式|状態方程式・随伴方程式・最適性条件|
|終端の考慮|部分積分による境界項|部分積分による境界項|
|考慮する変分|$\delta y = \varepsilon \eta(x)$ | $\delta x, \delta u, \delta \lambda$|

変分法では、$J[y]$を最小化する問題だったが、<br>
PMPでは $\dot{x}=f(x,u)$という制約が追加される。この制約を評価関数に組み込むために、ラグランジュ未定乗数 $\lambda(t)$を導入する。<br>
すると元の評価関数$J$にラグランジュ項を追加した「拡張評価関数」($\bar{J}$)を作ることになる。
$$
\bar{J}[x,u,\lambda] = \Phi(x(T)) + \int_0^T (L + \lambda^T(f-\dot{x}))dt
$$

変分法は「関数$y(x)$の変分 $\delta y$を考え、評価関数が停留する条件を求める」というものであり、<br>
PMPは「状態軌道$x(t)$と入力軌道 $u(t)$ に加え、状態方程式を制約として組み込むためのラグランジュ未定乗数 $\lambda(t)$を導入する。そして$x,u,\lambda$のすべての変分を考え、評価関数が停留する条件を求める」というものである。<br>

PMPは変分法を状態方程式という制約付き最適化問題へ拡張したものである。

## 順番

## 最適制御問題

最小化したい評価関数 $J[x,u]$を次とする。

$$
J[x,u] = \Phi(x(T)) + \int_0^T L(x(t), u(t), t) dt
$$

ここで
- $x(t)$ : 状態(state)
- $u(t)$ : 制御入力(control input)
- $T$ : 終端時刻

$J[x,u]$の各項目の意味を以下に説明する。

### 終端コスト $\Phi(x(T))$

評価関数を評価する期間の最終時刻$T$での状態(state)を評価する。<br>
例えば次のような状態
- 倒立揺子を真上で安定化させたい(角度=0, 角速度=0)
- ロボットの手先を目標位置姿勢にしたい ($x=X_T, y=Y_T, z=Z_T, \alpha = \alpha_T, \beta=\beta_T, \gamma=\gamma_T$)

具体的には、倒立揺子の状態を $x = [\theta, \omega]$ とし、最終時刻での状態を $x(T) = [\theta(T), \omega(T)]$、そこでの目標状態を$x_{ref} =  [\theta_T, \omega_T]$とするなら、

$$
\Phi(x(T)) = (x(T) - x_{ref})^T \mathbf{Q}_T (x(T) - x_{ref}) = (\theta(T) - \theta_{T}, \omega(T) - \omega_{T})^T \mathbf{Q}_T (\theta(T) - \theta_{T}, \omega(T) - \omega_{T})
$$
となり、これは最終状態がどれくらい目標状態に近いかを評価する。

$\mathbf{Q}_T$は通常、対称な正定値(または半正定値)行列であり、各状態の重みを表す。<br>
例えば以下のような行列である。
$$
\begin{bmatrix} 10 & 0 \\ 0 & 50 \end{bmatrix}, \space
\begin{bmatrix} 10 & 0 \\ 0 & 0 \end{bmatrix}
$$

2次形式であるため、スカラーであることに注意する。

### ランニングコスト $\int_0^T L(x,u,t) dt$

$\int_0^T L(x,u,t) dt$は区間[0,T]まで終端時刻に至るまでの状態$x$と制御入力$u$を評価する。
例えば、
$$L=x^T Q x + u^T R u$$
である。これは、「状態誤差を小さくしたい」、「制御入力をつかいすぎないようにしたい」という目的を示している。

また、目標状態 $x_{ref}$ を考慮するなら、以下の $L$ となり、この場合は 状態誤差($x-x_{ref}$)を小さくしたいという目的にになる。
$$
L=(x - x_{ref})^T Q (x- x_{ref}) + u^T R u
$$

そして2次形式で表しているため、$L$はスカラーであることに注意する。

### 制約の考慮

状態$x$は物理法則に従って変化するため、自由に選ぶことはできない。例えば時刻 k のとき 位置:0, 速度:0で、時刻 k+1 で位置:0, 速度:100などは物理法則に反している。<br>
例えばロボットでは、質量・慣性・重力などを考慮した運動方程式に従って運動するため、状態は入力による決まるため任意には決められない。<br>

倒立揺子である場合、次のような状態方程式になる。
$$
\dot{x} = \begin{bmatrix} \dot{p} \\ \ddot{p} \\ \dot{\theta} \\ \ddot{\theta} \end{bmatrix} = f(x,u)
$$

また、初期状態として、$x(0) = x_0$ という初期条件があり、これはスタート地点が固定されていることを意味してる。

PMPは次の最適制御問題を解く

$$
\min\limits_{x(t), u(t)} \space J = \Phi(x(T)) + \int_0^T L (x,u,t) dt \\
\text{subject to} \space \dot{x} = f(x,u,t), x(0) = x_0
$$

これは、「初期状態を$x_0$とする状態方程式に従って動く状態軌道 $x(t)$ と入力軌道 $u(t)$ の中から、評価関数 $J$ を最小にするものを求める」という問題である。

上記は数学的な見方として$x(t), u(t)$ を未知関数として扱っているが、状態方程式と初期条件が与えられれば、入力軌道 $u(t)$ を決めることで状態軌道 $x(u)$ が市井に決まる。そのため工学では、「最適な入力軌道$u(t)$を求める問題」と説明されることがある。

このノートでは、数学的な導出に従い、状態軌道$x(t)$と入力軌道$u(t)$の両方を未知関数として扱う。

ここまでは、最適制御問題の定義である。次に、この状態方程式という制約を評価関数へ組み込むため、ラグランジュ未定乗数を導入し、拡張評価関数を定義する。

## 状態方程式を評価関数へ組み込む

変分法では評価関数$J[y]$を停留させる関数を$\delta J=0$ の条件から求めていた。この時は制約条件なしであった。

最適制御問題では状態方程式 $\dot{x} = f(x,u,t)$ が制約となるため、これを評価関数へラグランジュの未定乗数法のように組み込む必要がある。

ラグランジュの未定乗数法では、$\min_x f(x) \space \text{subject to} \space g(x) = 0$であり、$\bar{f}(x,\lambda) = f(x) + \lambda g(x)$ という拡張目的関数を作った。ラグランジュの未定乗数法は 制約条件 $g(x)=0$ をラグランジュ未定乗数$\lambda$によって評価関数へ組み込み、制約付き最適化問題を、拡張目的関数の停留条件を求める問題へ書き換える。<br>

PMPでは制約である状態方程式 $\dot{x} = f(x,u,t)$ を $f(x,u,t) - \dot{x} = 0 $とすると、これは ラグランジュの未定乗数法における $g(x) = 0$ と同様であるため、ラグランジュ未定乗数 $\lambda(t)$ を制約条件 $f(x,u,t) - \dot{x} = 0 $ にかけ、その結果を評価関数へ加える。

状態方程式は一般に複数の状態変数から構成される。つまり状態 $x$は
$$
x = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{bmatrix}
$$
となるため、 $f(x,u,t) - \dot{x}$も同じ次元の縦ベクトルになる。
ラグランジュ未定乗数$\lambda(t)$ も

$$
\lambda = \begin{bmatrix} \lambda_1 \\ \lambda_2 \\ \vdots \\ \lambda_n \end{bmatrix}
$$

であるため、$\lambda^T(f(x,u,t)-\dot{x})$ を評価関数へ組み込む。<br>
状態方程式は評価期間[0,T]のすべてで満たす必要がある。そのため一点だけを評価する終端コストではなく、ランニングコストである時間全体を評価する積分項へ組み込み、次の拡張評価関数 $\bar{J}$を定義する。

$$
\bar{J}[x,u,\lambda] = \Phi(x(T)) + \int_0^T \left( L(x,u,t) + \lambda^T (f(x,u,t) - \dot{x})\right) dt
$$

状態方程式を満たす軌道では$(f(x,u,t)-\dot{x}) = 0$となるため、目的は最初と同じ $J = \Phi + \int L dt$ を最小化することと同一である。<br>
これは、状態方程式を満たす軌道に対しては $\bar{J} = J$ であるということになる。 
したがって、拡張評価関数$\bar{J}$は元の目的関数を変更したものではなく、状態方程式という制約を評価関数へ組み込むために導入した関数である。

この拡張評価関数に対して変分を取り、停留条件を求めることで、PMPの基本方程式を導出する。

-------------------------------------------------
memo 

$$
L_x = \frac{\partial L}{\partial x}, \space L_u = \frac{\partial L}{\partial u}
$$

例えば、$L=x^T Q_T x + u^T R u$なら<br>
$L_x = 2 \ Q \ x$, $L_u = 2 \ R \ u$ となる。
$Q, R$ それぞれ正方行列であり、$x, u$は縦ベクトルであるため、$L_x$は$x$方向に対する勾配ベクトル、$L_u$は$u$方向に対する勾配ベクトルである。

$$
f_x = \frac{\partial f}{\partial x}, \space f_u = \frac{\partial f}{\partial u}
$$

$f = [f_1, \cdots , f_n]^T $で、 $x=[x_1, \cdots, x_n]^T$ であるため、$f_x$は以下のヤコビアン行列(或いは単にヤコビアン)になる。

$$
\frac{\partial f}{\partial x} =
\begin{bmatrix}
\frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_n}{\partial x_1}\\
\vdots & \ddots & \vdots \\
\frac{\partial f_n}{\partial x_1} & \cdots & \frac{\partial f_n}{\partial x_n}
\end{bmatrix}
$$

$$
\Phi_x = \frac{\partial \Phi}{\partial x}
$$

例えば $\Phi = (x - x_{ref})^T \ Q_T \ (x - x_{ref})$　なら、 $\Phi_x = 2 \ Q_T \ (x-x_{ref})$ になる。<br>
これは、終端コストの$x$方向に関する偏微分で勾配ベクトルになる。

それぞれには以下の役割がある。
- $L_x$ : ランニングコストが状態でどのように変わるか
- $f_x$ : 状態方程式が状態でどのように変わるか
- $\Phi_x$ : 終端コストが終端状態でどのように変わるか

-------------------------------------------



変分法では$y(x) + \varepsilon \eta(x)$を考えたのと同様に、PMPでは状態 $x$、入力 $u$、ラグランジュ未定乗数 $\lambda$をそれぞれ以下のように微小変化させる。
$$
x + \varepsilon \delta x , \space u + \varepsilon \delta u, \space \lambda + \varepsilon \delta \lambda
$$

ここで次の補足を加える。

変分法では変分$\varepsilon \eta(x)$による近傍の変化量を以下のように記述した。
$$
J[y+\varepsilon \eta(x)] = J[y] + \delta J \varepsilon + O(\varepsilon^2)
$$
ここからJの一次変化 $\delta J$は次のようにも導出できる。

$$
\begin{array}{ll}
\delta J & = \left. \dfrac{d}{d \varepsilon} J[y+\varepsilon \eta(x)] \ \right|_{\varepsilon=0} \\
& = \left. \dfrac{d}{d \varepsilon} \left(  J[y] + \delta J \varepsilon + O(\varepsilon^2) \right) \ \right|_{\varepsilon=0} \\
& = \left. \left( 0 + \delta J + O(\varepsilon) \right) \ \right|_{\varepsilon=0} \\
& = \delta J
\end{array}
$$

$J[y]$は定数であり、$O(\varepsilon^2)=a\varepsilon^2 + b\varepsilon^3 \cdots$のような関数である。なお  $O(\varepsilon^2)$には定数は入らない。

つまり、評価関数 $J[y + \varepsilon \eta]$ に関して、$\varepsilon$へ微分して、$\varepsilon=0$における一次変化を求めたものが$\delta J$である。このように、関数へ微小な変化を与え、その一次変化を求める操作を「変分を取る」と呼ぶ。

そして、$\delta$という記号を以下の操作を行うこととして定義する。
$$
\delta = \left. \frac{d}{d \varepsilon} \right|_{\varepsilon=0}
$$


#### Step1 : 拡大評価関数

拡大評価関数 $\bar{J}$は以下である。

$$
\bar{J}[x,u,\lambda] = \Phi(x(T)) + \int_0^T \left( L(x,u,t) + \lambda^T (f(x,u,t) - \dot{x})\right) dt
$$

#### Step2 : 微小変化を代入した拡大評価関数

状態 $x$、入力 $u$、ラグランジュ未定乗数 $\lambda$をそれぞれ以下のように微小変化させる。
$$
\begin{array}{l}
x \rightarrow x + \varepsilon \delta x \\
u \rightarrow u + \varepsilon \delta u \\
\lambda \rightarrow  \lambda + \varepsilon \delta \lambda
\end{array}
$$

終端状態$x(T)$は以下となる。
$$
x(T) \rightarrow x(T) + \varepsilon \delta x(T)
$$

速度項 $\dot{x}$は以下となる。
$$
\dot{x} \rightarrow \frac{d}{dt}\left(x + \varepsilon \delta x \right) \rightarrow \dot{x} + \varepsilon \delta \dot{x}
$$

したがって、微小変化を代入した拡張評価関数 $\bar{J}$は以下となる。

$$
\begin{aligned}
&\bar{J}
\left[
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
\lambda+\varepsilon\delta\lambda
\right]
\\
&=
\Phi
\left(
x(T)+\varepsilon\delta x(T)
\right)
\\
&\quad+
\int_0^T
\Bigg[
L
\left(
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
t
\right)
\\
&\qquad+
\left(
\lambda+\varepsilon\delta\lambda
\right)^T
\Bigg(
f
\left(
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
t
\right)
-
\left(
\dot{x}+\varepsilon\delta\dot{x}
\right)
\Bigg)
\Bigg]dt
\end{aligned}
$$


#### Step 3 : 変分 $\delta$ の適用

変分 $\delta$ とは以下の操作を行うものであった。
$$
\delta = \left. \frac{d}{d \varepsilon} \right|_{\varepsilon=0}
$$ 

Step 2で求めた微小変化を代入した拡張評価関数 $\bar{J}$の変分を取る。<br>
左辺は以下となる。

$$
\begin{aligned}
\delta\bar{J}
&=
\left.
\frac{d}{d\varepsilon}
\bar{J}
\left[
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
\lambda+\varepsilon\delta\lambda
\right]
\right|_{\varepsilon=0}
\end{aligned}
$$

右辺は以下のようになる。

$$
\begin{aligned}
\delta\bar{J}&= \frac{d}{d\varepsilon} \Bigg[ \Phi \left(x(T)+\varepsilon\delta x(T) \right) \\
&\qquad+\int_0^T \Bigg( L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u, \ t \right) \\
&\qquad+\left( \lambda+\varepsilon\delta\lambda \right)^T \Bigg( f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u, \  t \right)-\left( \dot{x}+\varepsilon\delta\dot{x} \right) \Bigg) \Bigg) dt \Bigg]
\Bigg|_{\varepsilon=0}
\end{aligned}
$$

#### Step 4 項目の分割

上式は以下のように表現することができる。

$$
\delta \bar{J} = \left. \frac{d}{d \varepsilon} \left[ \Phi(x(T) + \varepsilon \delta x(T)) + \int_0^T A(\varepsilon, t)  dt \right] \right|_{\varepsilon = 0}
$$

ここで、$A(\varepsilon, t)$は以下である。
$$
\begin{aligned}
A(\varepsilon, t) =& L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u, \ t \right) \\
&\qquad+\left( \lambda+\varepsilon\delta\lambda \right)^T \Bigg( f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u, \  t \right)-\left( \dot{x}+\varepsilon\delta\dot{x} \right) \Bigg)
\end{aligned}
$$

変分を項目ごとに分けると、次のようになる。

$$
\begin{aligned}
\delta \bar{J} &= \left. \frac{d}{d \varepsilon} \Phi(x(T) + \varepsilon \delta x(T)) \right|_{\varepsilon=0} \\ &\quad + \frac{d}{d \varepsilon} \left. \int_0^T A(\varepsilon, t)  dt  \right|_{\varepsilon = 0}
\end{aligned}
$$

積分は$t$に関して行われる。$\varepsilon$は$t$に関しては単なるパラメータであるので、積分後も$\varepsilon$の関数として$A(\varepsilon)$は残る。そのため、その後に$\varepsilon$ で微分できる。また、積分区間[0,T]が$\varepsilon$に依存しないため、微分と積分の順序を交換できる。よって、$\delta \bar{J}$は次のようになる。

$$
\begin{aligned}
\delta \bar{J} &= \left. \frac{d}{d \varepsilon} \Phi(x(T) + \varepsilon \delta x(T)) \right|_{\varepsilon=0} \\ &\quad +  \left. \int_0^T \frac{d}{d \varepsilon} A(\varepsilon, t)  \right|_{\varepsilon = 0} dt
 \end{aligned}
$$


上記の微分と積分の順序を入れ替えることが出来ることを$A(\varepsilon,t) = \varepsilon t + t^2$ と簡単な例に置き換えて確認する。

$$
\int_0^T A(\varepsilon,t) dt = \int_0^T (\varepsilon t + t^2) dt = \varepsilon \frac{T^2}{2} + \frac{T^3}{3}
$$
この結果から分かるように、$\varepsilon$ は$t$のパラメータであり、積分後は変数$t$が定数$T$となるため$A(\varepsilon)$の関数が残る。

これを $\varepsilon$で微分すると、$T^2/2$となる。
$$
\frac{d}{d \varepsilon}\left( \varepsilon \frac{T^2}{2} + \frac{T^3}{3} \right) = \frac{T^2}{2}
$$

一方積分の中で先に微分して、その後その結果を積分すると、同じく$T^2/2$となる。
$$
\frac{\partial A(\varepsilon, t)}{\partial \varepsilon} = t \rightarrow \int_0^T t dt = \frac{T^2}{2}
$$ 


#### Step 5 : 終端コスト $\Phi$ の変分

Step 4 で導出した式の第1項の変分計算を行う。

$$
\left. \frac{d}{d \varepsilon} \Phi(x(T) + \varepsilon \delta x(T)) \right|_{\varepsilon=0}
$$

チェインルールを適用すると、$\varepsilon$で微分を行うため、$\Phi(x(T) + \varepsilon \delta x(T))$は変数$x(T)$の関数であり、$x(T) + \varepsilon \delta x(T)$は$\varepsilon$の関数であると捉えると。
$$
\frac{d}{d \varepsilon} \Phi(x(T) + \varepsilon \delta x(T)) = \frac{\partial \Phi(x(T))}{\partial x(T)} \frac{d}{d \varepsilon}(x(T) + \varepsilon \delta x(T)) = \frac{\partial \Phi(x(T)}{\partial x} \delta x(T) = \Phi_x \delta x(T)  
$$

最後に $\varepsilon=0$ として$\delta \Phi$を求める。
$$
\delta \Phi = \Phi_x^T \delta x(T)
$$

なおここでは、$\Phi_x, \delta x(T)$を縦ベクトルとしているため、$\Phi_x^T$としている。

$$
\Phi_x = \begin{bmatrix}
\partial \Phi / \partial x_1 \\
\vdots \\
\partial \Phi / \partial x_n \\
\end{bmatrix
$$}



終端コストを変分する。

状態を

$$
\delta \Phi(x(T)) = 
$$
